# Data Cleaning & Transformation

In [ ]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
.appName('Olist_Data')\
.getOrCreate()

In [ ]:
hdfs_path='/data/olist/'

In [ ]:
customers_df=spark.read.csv(hdfs_path + 'olist_customers_dataset.csv',header=True,inferSchema=True)
orders_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv',header=True,inferSchema=True)
order_item_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv',header=True,inferSchema=True)
payments_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv',header=True,inferSchema=True)
reviews_df = spark.read.csv(hdfs_path + 'olist_order_reviews_dataset.csv',header=True,inferSchema=True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv',header=True,inferSchema=True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv',header=True,inferSchema=True)
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv',header=True,inferSchema=True)
category_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv',header=True,inferSchema=True)


In [ ]:
# Identify missing values
from pyspark.sql.functions import * 
def missing_values(df,df_name):
    print(f'Missing value in {df_name}:')
    df.select([count(when(col(c).isNull(),1)).alias(c) for c in df.columns]).show()

In [ ]:
missing_values(customers_df,'customer')

In [ ]:
missing_values(orders_df,'order')

In [ ]:
missing_values(order_item_df,'order_item')

In [ ]:
# Handle Misiing Values
orders_df_cleaned=orders_df.na.drop(subset=['customer_id','order_id','order_status'])
orders_df_cleaned.show(5)

In [ ]:
# filling missing values
cleaned_df=orders_df_cleaned.fillna({'order_delivered_customer_date':'9999-12-31'})
cleaned_df.show(10)


# Impute missing values

In [ ]:
# creating null df of payments
payments_null_df=payments_df.withColumn('payment_value',when(col('payment_value')!=99.33,col('payment_value')).otherwise(lit(None)))
payments_null_df.show(5)

In [ ]:
# intitaizing Imputer
from pyspark.ml.feature import Imputer
imputer=Imputer(inputCols=['payment_value'],outputCols=['payment_imputed_value']).setStrategy('median')
cleaned_payment_df=imputer.fit(payments_null_df).transform(payments_null_df)
cleaned_payment_df.show(5)

# Standardizing The Format

In [ ]:
def print_schema(df,df_name):
    print(f'Schema of the {df_name} is :')
    df.printSchema()

In [ ]:
print_schema(orders_df,'orders')

In [ ]:
print_schema(customers_df,'customers')

In [ ]:
cleaned_order_df=cleaned_df.withColumn('order_purchase_timestamp',to_date(col('order_purchase_timestamp')))
cleaned_order_df.show(5)

In [ ]:
payments_df.show(10)

In [ ]:
# Cleaning PaymentType
cleaned_payment_df=cleaned_payment_df.withColumn('payment_type',when(col('payment_type')=='boleto','Bank Transfer')
                                             .when(col('payment_type')=='credit_card','credit card')
                                             .when(col('payment_type')=='debit_card','debit card')
                                             .otherwise('other'))

In [ ]:
cleaned_payment_df.show(15)

In [ ]:
# More data cleaning
print_schema(customers_df,'customer')
customers_df.show(1)

In [ ]:
customers_cleaned_df=customers_df.withColumn('customer_zip_code_prefix',col('customer_zip_code_prefix').cast('string'))

In [ ]:
print_schema(customers_cleaned_df,'customers')
customers_df.show(5)

# Remove Duplicate Records

In [ ]:
customers_cleaned_df=customers_cleaned_df.dropDuplicates(['customer_id'])

# Data Transormation

In [ ]:
order_with_details=cleaned_order_df.join(order_item_df,'order_id','left')\
            .join(cleaned_payment_df,'order_id','left')\
            .join(customers_cleaned_df,'customer_id','left')

In [ ]:
# Heavy
order_with_details.show(1,vertical=True)

In [ ]:
order_with_total_value=order_with_details.groupBy('order_id')\
        .agg(sum('payment_value')).alias('Total_order_Value')

In [ ]:
order_with_total_value.show(5)

In [ ]:
# Delivery Time
delivery_df=order_with_details.select('order_id','order_purchase_timestamp','order_delivered_customer_date')

In [ ]:
from pyspark.sql.functions import datediff,to_date
delivery_detail_df=delivery_df.withColumn('delivery_time',datediff(col('order_delivered_customer_date'),col('order_purchase_timestamp')))
delivery_detail_df.show(5)

# Advanced Transformation

In [ ]:
quantiles=order_item_df.approxQuantile('price',[0.01,0.99],0.0)
low_cutoff,high_cutoff=quantiles[0],quantiles[1]

In [ ]:
order_item_df.select('price').summary().show()

In [ ]:
low_cutoff,high_cutoff

In [ ]:
order_item_cleaned_df=order_item_df.filter((col('price')>=low_cutoff) & (col('price')<=high_cutoff))

In [ ]:
# Cleaned Data
order_item_cleaned_df.show(10)

In [ ]:
cleaned_payment_df.select('payment_installments').summary().show()

In [ ]:
products_df.show(3)

In [ ]:
# Product Transformation
products_cleaned_df=products_df.withColumn('product_size_category',(when(col('product_weight_g') < 500,'Small'))
                                           .when(col('product_weight_g').between(500 , 2000),'Medium')
                                           .when(col('product_weight_g') >2000,'Large')
                                          )

In [ ]:
products_cleaned_df.show(2,vertical=True)

# Total Revenue Per Customer

In [ ]:
Total_Revenue_Per_Customer=order_with_details.groupBy('customer_id').agg(sum('price').alias('Total Revenue Per Seller')).orderBy('Total Revenue Per Seller',ascending=False)
Total_Revenue_Per_Customer.show(10)

In [ ]:
!hadoop fs -mkdir /data/olist_processed

# Data Storing In HDFS

In [ ]:
!hadoop fs -ls /data

In [ ]:
order_with_details.write.mode('overwrite').parquet('/data/olist_processed/cleaned_data_parquet')

In [ ]:
products_cleaned_df.write.mode('overwrite').parquet('/data/olist_processed/product_cleaned_data.parquet')

In [ ]:
spark.sql("""
CREATE EXTERNAL TABLE cleaned_orders (
    product_id STRING,
    product_category_name STRING,
    product_name_lenght INT,
    product_description_lenght INT,
    product_photos_qty INT,
    product_weight_g INT,
    product_length_cm INT,
    product_height_cm INT,
    product_width_cm INT,
    product_size_category STRING
)
STORED AS PARQUET
LOCATION '/data/olist_processed/product_cleaned_data.parquet'
""")
